# 🎬 GRU를 활용한 IMDB 감성 분류기

## GRU (Gated Recurrent Unit) 구조

LSTM의 간소화 버전 — **게이트 2개**로 장기 의존성 처리

$$z_t = \sigma(W_z [h_{t-1}, x_t])  \quad \text{(Update gate: 얼마나 업데이트할지)}$$
$$r_t = \sigma(W_r [h_{t-1}, x_t])  \quad \text{(Reset gate: 이전 기억 얼마나 버릴지)}$$
$$\tilde{h}_t = \tanh(W[r_t \odot h_{t-1}, x_t])  \quad \text{(후보 hidden state)}$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

| | LSTM | GRU |
|--|------|-----|
| **게이트 수** | 4개 (f, i, g, o) | 2개 (z, r) |
| **파라미터** | 많음 | 적음 (~25% 절약) |
| **속도** | 느림 | 빠름 |
| **성능** | 일반적으로 우수 | 짧은 시퀀스에서 LSTM과 유사 |

## 파이프라인
```
IMDB 데이터 로드 → 토큰화/패딩 → Embedding → GRU → Dense → 이진 분류
```

In [ ]:
# 필요 라이브러리 설치 확인
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'torchtext',
                'datasets', 'transformers', '--quiet'], check=False)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np
import random
import re

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## 1️⃣ IMDB 데이터 로드 (Keras 내장 데이터셋)

In [ ]:
# Keras의 IMDB 데이터셋 사용 (전처리된 정수 시퀀스)
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB  = 10000   # 사용할 어휘 크기
MAX_LEN    = 200     # 시퀀스 최대 길이

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_VOCAB)
print(f'Train: {len(x_train)}개  /  Test: {len(x_test)}개')
print(f'클래스: 0=부정, 1=긍정')
print(f'학습 긍정 비율: {y_train.mean()*100:.1f}%')

# 길이 통계
lengths = [len(x) for x in x_train]
print(f'\n리뷰 길이 통계:')
print(f'  평균: {np.mean(lengths):.1f} / 최대: {np.max(lengths)} / 중앙값: {np.median(lengths):.0f}')

In [ ]:
# 패딩 (앞쪽 잘라내기 — 리뷰 뒷부분이 더 중요한 경향)
x_train_pad = pad_sequences(x_train, maxlen=MAX_LEN, padding='pre', truncating='pre')
x_test_pad  = pad_sequences(x_test,  maxlen=MAX_LEN, padding='pre', truncating='pre')

print(f'패딩 후 shape: {x_train_pad.shape}  → (샘플수, 시퀀스 길이)')

# 원본 리뷰 확인
word_index = imdb.get_word_index()
inv_index  = {v+3: k for k, v in word_index.items()}
inv_index.update({0: '<PAD>', 1: '<BOS>', 2: '<UNK>', 3: '<UNK>'})

def decode_review(seq):
    return ' '.join(inv_index.get(i, '?') for i in seq if i > 0)

print(f'\n[예시 리뷰] 레이블={y_train[0]}')
print(decode_review(x_train[0][:50]) + '...')

## 2️⃣ PyTorch 데이터셋 & 데이터로더

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


BATCH_SIZE = 64

# 학습/검증 분리 (학습 20000 / 검증 5000)
train_ds   = IMDBDataset(x_train_pad[:20000], y_train[:20000])
valid_ds   = IMDBDataset(x_train_pad[20000:], y_train[20000:])
test_ds    = IMDBDataset(x_test_pad,          y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train: {len(train_ds)} / Valid: {len(valid_ds)} / Test: {len(test_ds)}')

## 3️⃣ GRU 감성 분류 모델

```
Embedding(10000, 128)
     ↓
GRU(128→256, layers=2, bidirectional, dropout=0.3)
     ↓
  [마지막 hidden: fwd + bwd 연결]
     ↓
Linear(512→128) → ReLU → Dropout
     ↓
Linear(128→1) → Sigmoid → 이진 분류
```

In [ ]:
class GRUSentimentClassifier(nn.Module):
    def __init__(self,
                 vocab_size   = 10000,
                 embed_dim    = 128,
                 hidden_dim   = 256,
                 num_layers   = 2,
                 dropout      = 0.3,
                 bidirectional= True,
                 pad_idx      = 0):
        super().__init__()
        self.bidirectional = bidirectional

        # ① Embedding
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_drop = nn.Dropout(dropout)

        # ② GRU (핵심 레이어)
        self.gru = nn.GRU(
            input_size    = embed_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = dropout if num_layers > 1 else 0.0
        )

        # ③ 분류기
        gru_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(gru_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)   # 이진 분류 → Sigmoid 따로
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.05, 0.05)
        self.embedding.weight.data[0].zero_()  # PAD = 0

    def forward(self, x):
        # x: (B, L)
        emb = self.embed_drop(self.embedding(x))     # (B, L, E)
        out, h_n = self.gru(emb)                     # h_n: (num_layers*D, B, H)

        # 마지막 레이어의 순/역방향 hidden 연결
        if self.bidirectional:
            # h_n[-2]: 마지막 레이어 순방향
            # h_n[-1]: 마지막 레이어 역방향
            rep = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (B, 2H)
        else:
            rep = h_n[-1]                               # (B, H)

        return self.classifier(rep).squeeze(1)          # (B,)


model = GRUSentimentClassifier(
    vocab_size    = MAX_VOCAB,
    embed_dim     = 128,
    hidden_dim    = 256,
    num_layers    = 2,
    dropout       = 0.3,
    bidirectional = True,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'총 파라미터:  {total_params:,}')
print(f'학습 파라미터: {train_params:,}')
print()
print(model)

## 4️⃣ 학습 설정 & 루프

In [ ]:
criterion = nn.BCEWithLogitsLoss()   # Sigmoid + BCE 통합 (수치 안정)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5, verbose=True
)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        preds  = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        preds  = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return total_loss / total, correct / total

print('학습 준비 완료!')

In [ ]:
EPOCHS = 10
best_val_acc = 0.0
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8}")
print('=' * 58)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = evaluate(model, valid_loader, criterion)
    scheduler.step(vl_acc)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    mark = ' ★' if vl_acc > best_val_acc else ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '/tmp/best_gru_imdb.pt')

    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc*100:>8.2f}% | {vl_loss:>9.4f} | {vl_acc*100:>7.2f}%{mark}")

print(f'\n최고 Validation Accuracy: {best_val_acc*100:.2f}%')

## 5️⃣ 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Loss Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc')
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc')
axes[1].set_title('Accuracy Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gru_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('학습 곡선 저장 완료')

## 6️⃣ 테스트셋 최종 평가

In [ ]:
model.load_state_dict(torch.load('/tmp/best_gru_imdb.pt', map_location=device))
test_loss, test_acc = evaluate(model, test_loader, criterion)

print('=' * 40)
print(f'  Test Loss    : {test_loss:.4f}')
print(f'  Test Accuracy: {test_acc*100:.2f}%')
print('=' * 40)

# 혼동 행렬
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        logits = model(x.to(device))
        preds  = (torch.sigmoid(logits) > 0.5).int().cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.int().tolist())

print('\n[Classification Report]')
print(classification_report(all_labels, all_preds,
                             target_names=['부정(0)', '긍정(1)']))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['부정', '긍정'],
            yticklabels=['부정', '긍정'], ax=ax)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('예측'); ax.set_ylabel('실제')
plt.tight_layout()
plt.show()

## 7️⃣ 새 리뷰 감성 예측

In [ ]:
@torch.no_grad()
def predict_sentiment(review_text, model, word_index, max_len=200):
    model.eval()
    # 전처리
    review_text = re.sub(r'[^a-zA-Z\s]', '', review_text.lower())
    words = review_text.split()
    ids   = [word_index.get(w, 2) + 3 for w in words]  # +3: special token offset
    ids   = ids[-max_len:]  # 최대 길이 자르기
    # 패딩
    if len(ids) < max_len:
        ids = [0] * (max_len - len(ids)) + ids
    x      = torch.tensor([ids], dtype=torch.long, device=device)
    logit  = model(x)
    prob   = torch.sigmoid(logit).item()
    label  = '긍정 😊' if prob > 0.5 else '부정 😞'
    return label, prob


reviews = [
    "This movie was absolutely fantastic! The acting was superb and the story was deeply moving.",
    "Terrible waste of time. The plot made no sense and the acting was awful. I hated it.",
    "An average film. Some parts were good but others were quite boring and predictable.",
    "One of the best films I have seen in years. Brilliant cinematography and performances.",
    "The movie started well but completely fell apart in the second half. Very disappointing.",
]

print('[새 리뷰 감성 예측]\n')
for review in reviews:
    label, prob = predict_sentiment(review, model, word_index)
    bar_p = '█' * int(prob * 20)
    bar_n = '░' * (20 - int(prob * 20))
    print(f'리뷰: "{review[:60]}..."')
    print(f'예측: {label}  (긍정 확률: {prob:.4f})')
    print(f'[{bar_p}{bar_n}]  긍정 {prob*100:.1f}% | 부정 {(1-prob)*100:.1f}%')
    print()

## 8️⃣ 실험: GRU vs LSTM 비교

같은 하이퍼파라미터로 GRU와 LSTM의 성능과 속도를 비교합니다.

In [ ]:
import time

class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim*2, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, 1)
        )
    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        rep = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.classifier(rep).squeeze(1)


results = {}
for name, cls in [('GRU', GRUSentimentClassifier), ('LSTM', LSTMSentimentClassifier)]:
    m   = cls().to(device)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.BCEWithLogitsLoss()
    params = sum(p.numel() for p in m.parameters())

    t0 = time.time()
    for _ in range(3):  # 3 epoch만 비교
        train_epoch(m, train_loader, opt, crit)
    elapsed = time.time() - t0
    _, val_acc = evaluate(m, valid_loader, crit)

    results[name] = {'acc': val_acc, 'time': elapsed, 'params': params}
    print(f'{name}:  파라미터={params:,}  /  3epoch 소요={elapsed:.1f}s  /  Val Acc={val_acc*100:.2f}%')

print(f"\nGRU 파라미터 절약: {(1 - results['GRU']['params']/results['LSTM']['params'])*100:.1f}%")
print(f"GRU 속도 향상:     {results['LSTM']['time']/results['GRU']['time']:.2f}x")